## 기본 예시: 프롬프트 + 모델 + 출력 파서

가장 기본적이고 일반적인 사용 사례는 prompt 템플릿과 모델을 함께 연결하는 것입니다. 이것이 어떻게 작동하는지 보기 위해, 각 나라별 수도를 물어보는 Chain을 생성해 보겠습니다.


In [7]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [9]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH01-Basic")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH01-Basic


## 프롬프트 템플릿의 활용

`PromptTemplate`

- 사용자의 입력 변수를 사용하여 완전한 프롬프트 문자열을 만드는 데 사용되는 템플릿입니다
- 사용법
  - `template`: 템플릿 문자열입니다. 이 문자열 내에서 중괄호 `{}`는 변수를 나타냅니다.
  - `input_variables`: 중괄호 안에 들어갈 변수의 이름을 리스트로 정의합니다.

`input_variables`

- input_variables는 PromptTemplate에서 사용되는 변수의 이름을 정의하는 리스트입니다.

In [2]:
from langchain_teddynote.messages import stream_response  # 스트리밍 출력
from langchain_core.prompts import PromptTemplate

`from_template()` 메소드를 사용하여 PromptTemplate 객체 생성


In [3]:
# template 정의
template = "{country}의 수도는 어디인가요?"

# from_template 메소드를 이용하여 PromptTemplate 객체 생성
prompt_template = PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='{country}의 수도는 어디인가요?')

In [4]:
# prompt 생성
prompt = prompt_template.format(country="대한민국")
prompt

'대한민국의 수도는 어디인가요?'

In [5]:
# prompt 생성
prompt = prompt_template.format(country="미국")
prompt

'미국의 수도는 어디인가요?'

In [11]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
)

## Chain 생성

### LCEL(LangChain Expression Language)

![lcel.png](./images/lcel.png)

여기서 우리는 LCEL을 사용하여 다양한 구성 요소를 단일 체인으로 결합합니다

```
chain = prompt | model | output_parser
```

`|` 기호는 [unix 파이프 연산자](<https://en.wikipedia.org/wiki/Pipeline_(Unix)>)와 유사하며, 서로 다른 구성 요소를 연결하고 한 구성 요소의 출력을 다음 구성 요소의 입력으로 전달합니다.

이 체인에서 사용자 입력은 프롬프트 템플릿으로 전달되고, 그런 다음 프롬프트 템플릿 출력은 모델로 전달됩니다. 각 구성 요소를 개별적으로 살펴보면 무슨 일이 일어나고 있는지 이해할 수 있습니다.


In [15]:
# prompt 를 PromptTemplate 객체로 생성합니다.
prompt = PromptTemplate.from_template("{topic} 에 대해 쉽게 설명해주세요.")

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

chain = prompt | model

In [16]:
chain

PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='{topic} 에 대해 쉽게 설명해주세요.')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x112e18050>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1170349d0>, root_client=<openai.OpenAI object at 0x113cb1d50>, root_async_client=<openai.AsyncOpenAI object at 0x112e19850>, model_name='gpt-4o-mini', temperature=0.1, model_kwargs={}, openai_api_key=SecretStr('**********'))

### invoke() 호출

- python 딕셔너리 형태로 입력값을 전달합니다.(키: 값)
- invoke() 함수 호출 시, 입력값을 전달합니다.

In [13]:
# input 딕셔너리에 주제를 '인공지능 모델의 학습 원리'으로 설정합니다.
input = {"topic": "인공지능 모델의 학습 원리"}

In [17]:
# prompt 객체와 model 객체를 파이프(|) 연산자로 연결하고 invoke 메서드를 사용하여 input을 전달합니다.
# 이를 통해 AI 모델이 생성한 메시지를 반환합니다.
chain.invoke(input)

# 아래처럼도 사용 가능 (위에 권장)
# chain.invoke("input 내용")

AIMessage(content='인공지능 모델의 학습 원리를 쉽게 설명하자면, 다음과 같은 단계로 나눌 수 있습니다.\n\n1. **데이터 수집**: 인공지능 모델은 학습하기 위해 많은 데이터를 필요로 합니다. 이 데이터는 이미지, 텍스트, 소리 등 다양한 형태일 수 있습니다.\n\n2. **데이터 전처리**: 수집한 데이터는 모델이 이해할 수 있는 형태로 가공해야 합니다. 예를 들어, 이미지의 크기를 조정하거나, 텍스트를 숫자로 변환하는 과정이 필요합니다.\n\n3. **모델 선택**: 학습할 모델을 선택합니다. 이는 신경망, 결정 트리, 서포트 벡터 머신 등 여러 종류가 있습니다. 각 모델은 특정한 문제에 더 적합할 수 있습니다.\n\n4. **학습**: 모델은 주어진 데이터를 바탕으로 패턴을 학습합니다. 이 과정에서 모델은 입력 데이터와 정답(라벨) 간의 관계를 파악하게 됩니다. 예를 들어, 고양이와 개의 이미지를 구분하는 모델은 각각의 특징을 학습합니다.\n\n5. **손실 함수**: 모델의 예측이 실제 정답과 얼마나 차이가 있는지를 측정하는 손실 함수를 사용합니다. 이 값을 최소화하는 방향으로 모델의 파라미터(가중치)를 조정합니다.\n\n6. **최적화**: 경량화된 알고리즘(예: 경사 하강법)을 사용하여 손실 함수를 최소화하는 방향으로 모델의 파라미터를 업데이트합니다. 이 과정을 여러 번 반복하면서 모델이 점점 더 정확해집니다.\n\n7. **검증**: 학습이 끝난 후, 모델의 성능을 평가하기 위해 새로운 데이터(검증 데이터)를 사용합니다. 이 데이터는 모델이 학습할 때 사용하지 않은 데이터로, 모델의 일반화 능력을 확인하는 데 중요합니다.\n\n8. **배포**: 모델이 충분히 학습하고 검증되면, 실제 환경에서 사용할 수 있도록 배포합니다. 이후에도 새로운 데이터로 모델을 업데이트하거나 개선할 수 있습니다.\n\n이러한 과정을 통해 인공지능 모델은 주어진 문제를 해결할 수 있는 능력을 갖추게 됩니다.', additional_kwargs={'ref

아래는 스트리밍을 출력하는 예시 입니다.

In [ ]:
# 스트리밍 출력을 위한 요청
answer = chain.stream(input)
# 스트리밍 출력
stream_response(answer)

### 출력파서(Output Parser)


In [18]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

Chain 에 출력파서를 추가합니다.

In [19]:
# 프롬프트, 모델, 출력 파서를 연결하여 처리 체인을 구성합니다.
chain = prompt | model | output_parser

In [20]:
# chain 객체의 invoke 메서드를 사용하여 input을 전달합니다.
input = {"topic": "인공지능 모델의 학습 원리"}
chain.invoke(input)

'인공지능 모델의 학습 원리를 쉽게 설명하자면, 다음과 같은 단계로 이해할 수 있습니다.\n\n1. **데이터 수집**: 인공지능 모델은 학습하기 위해 많은 데이터를 필요로 합니다. 이 데이터는 이미지, 텍스트, 소리 등 다양한 형태일 수 있습니다.\n\n2. **데이터 전처리**: 수집한 데이터는 모델이 이해할 수 있는 형태로 가공해야 합니다. 예를 들어, 이미지의 크기를 조정하거나, 텍스트를 숫자로 변환하는 등의 작업이 필요합니다.\n\n3. **모델 선택**: 학습할 모델을 선택합니다. 이는 신경망, 결정 트리, 서포트 벡터 머신 등 여러 가지 방법이 있을 수 있습니다. 각 모델은 특정한 문제에 더 적합할 수 있습니다.\n\n4. **학습**: 모델은 주어진 데이터를 바탕으로 패턴을 학습합니다. 이 과정에서 모델은 입력 데이터와 정답(라벨)을 비교하여 오차를 계산하고, 이 오차를 줄이기 위해 모델의 파라미터(가중치)를 조정합니다. 이 과정을 반복하면서 모델은 점점 더 정확한 예측을 할 수 있게 됩니다.\n\n5. **검증**: 학습이 끝난 후, 모델의 성능을 평가하기 위해 새로운 데이터(검증 데이터)를 사용합니다. 이 데이터는 모델이 학습할 때 사용하지 않은 데이터로, 모델이 실제 상황에서 얼마나 잘 작동하는지를 확인하는 데 사용됩니다.\n\n6. **튜닝**: 모델의 성능이 만족스럽지 않다면, 하이퍼파라미터 조정, 더 많은 데이터 수집, 또는 다른 모델을 시도하는 등의 방법으로 개선할 수 있습니다.\n\n7. **배포**: 최종적으로 학습된 모델은 실제 환경에 배포되어 사용됩니다. 사용자가 입력한 데이터를 바탕으로 예측을 하거나, 분류를 수행하는 등의 작업을 하게 됩니다.\n\n이러한 과정을 통해 인공지능 모델은 데이터를 기반으로 학습하고, 새로운 상황에서도 유용한 예측을 할 수 있게 됩니다.'

In [ ]:
# 스트리밍 출력을 위한 요청
answer = chain.stream(input)
# 스트리밍 출력
stream_response(answer)

### 템플릿을 변경하여 적용

- 아래의 프롬프트 내용을 얼마든지 **변경** 하여 테스트 해볼 수 있습니다.
- `model_name` 역시 변경하여 테스트가 가능합니다.

In [26]:
template = """
당신은 영어를 가르치는 10년차 영어 선생님입니다. 주어진 상황에 맞는 영어 회화를 작성해 주세요.
양식은 [FORMAT]을 참고하여 작성해 주세요.

#상황:
{question}

#FORMAT:
- 영어 회화:
- 한글 해석:
"""

# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다.
model = ChatOpenAI(model_name="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

In [27]:
# 체인을 구성합니다.
chain = prompt | model | output_parser

In [28]:
# 완성된 Chain을 실행하여 답변을 얻습니다.
print(chain.invoke({"question": "저는 식당에 가서 음식을 주문하고 싶어요"}))

- 영어 회화:
  - Waiter: Good evening! Welcome to our restaurant. How many people are you dining with?
  - You: Just one, please.
  - Waiter: Great! Here’s the menu. Can I get you something to drink while you look it over?
  - You: Yes, I’d like a glass of water, please.
  - Waiter: Sure! Are you ready to order, or do you need more time?
  - You: I think I’m ready. I’d like the grilled chicken with vegetables, please.
  - Waiter: Excellent choice! Would you like anything else with that, like a side or dessert?
  - You: No, that’s all for now. Thank you!
  - Waiter: You’re welcome! I’ll get that order in for you.

- 한글 해석:
  - 웨이터: 안녕하세요! 저희 식당에 오신 것을 환영합니다. 몇 분이서 오셨나요?
  - 당신: 저 혼자입니다.
  - 웨이터: 좋습니다! 여기 메뉴입니다. 주문하시기 전에 음료수 드릴까요?
  - 당신: 네, 물 한 잔 주세요.
  - 웨이터: 알겠습니다! 주문할 준비가 되셨나요, 아니면 더 필요하신가요?
  - 당신: 저는 준비가 된 것 같아요. 구운 치킨과 야채를 주문할게요.
  - 웨이터: 훌륭한 선택입니다! 사이드나 디저트 같은 다른 것도 필요하신가요?
  - 당신: 아니요, 지금은 그게 다입니다. 감사합니다!
  - 웨이터: 천만에요! 주문을 넣어드릴게요.


In [ ]:
# 완성된 Chain을 실행하여 답변을 얻습니다.
# 스트리밍 출력을 위한 요청
answer = chain.stream({"question": "저는 식당에 가서 음식을 주문하고 싶어요"})
# 스트리밍 출력
stream_response(answer)

In [25]:
# 이번에는 question 을 '미국에서 피자 주문'으로 설정하여 실행합니다.
# 스트리밍 출력을 위한 요청
answer = chain.stream({"question": "미국에서 피자 주문"})
# 스트리밍 출력
stream_response(answer)

- 영어 회화:
A: Hi there! I’d like to place an order for a pizza, please.
B: Of course! What size would you like?
A: I’ll have a large, please. Can I get half pepperoni and half veggie?
B: Absolutely! Would you like any extra toppings?
A: Yes, could you add mushrooms to the veggie side?
B: Sure! Anything to drink with that?
A: Yes, I’ll take a large soda, please.
B: Great! Your total comes to $25. Would you like to pay with cash or card?
A: I’ll pay with a card. 
B: Perfect! Your order will be ready in about 30 minutes. Thank you!
A: Thank you!

- 한글 해석:
A: 안녕하세요! 피자 주문하고 싶어요.
B: 물론이죠! 어떤 사이즈로 하시겠어요?
A: 큰 사이즈로 주세요. 반은 페퍼로니, 반은 채소로 해주세요.
B: 알겠습니다! 추가 토핑은 필요하신가요?
A: 네, 채소 쪽에 버섯을 추가해 주세요.
B: 좋습니다! 음료는 필요하신가요?
A: 네, 큰 사이즈의 탄산음료 하나 주세요.
B: 좋습니다! 총 금액은 25달러입니다. 현금으로 결제하시겠어요, 카드로 하시겠어요?
A: 카드로 결제할게요.
B: 완벽합니다! 주문은 30분 정도 후에 준비될 거예요. 감사합니다!
A: 감사합니다!